In [8]:
# Import required libraries
import pandas as pd
import numpy as np
import re
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Set random seed for reproducibility
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

# Define paths
BASE_PATH = Path('../steam_data_20251208')
REVIEWS_PATH = BASE_PATH / 'reviews'
METADATA_PATH = BASE_PATH / 'games_metadata_20251208.csv'
PLAYER_COUNT_PATH = BASE_PATH / 'player_count_history.csv'

## 1. Load and Explore Data

In [9]:
# Load games metadata
games_df = pd.read_csv(METADATA_PATH)
print(f"Total games in metadata: {len(games_df)}")
print(f"\nColumns: {games_df.columns.tolist()}")
print("\nGenres distribution:")
print(games_df['primary_genre_query'].value_counts())

Total games in metadata: 120

Columns: ['appid', 'primary_genre_query', 'name', 'type', 'release_date', 'is_free', 'developers', 'publishers', 'genres', 'short_description', 'num_reviews_in_query', 'review_score', 'review_score_desc', 'total_positive', 'total_negative', 'total_reviews']

Genres distribution:
primary_genre_query
sports                    10
horror                    10
science_fiction           10
exploration_open_world    10
anime                     10
survival                  10
action_fps                10
hidden_object             10
rpg_action                10
casual                    10
puzzle_matching           10
visual_novel              10
Name: count, dtype: int64


In [10]:
# Load player count history for popularity analysis
player_count_df = pd.read_csv(PLAYER_COUNT_PATH)
print(f"Player count records: {len(player_count_df)}")
print(f"\nColumns: {player_count_df.columns.tolist()}")
print("\nSample data:")
player_count_df.head()

Player count records: 34160

Columns: ['timestamp', 'appid', 'player_count']

Sample data:


,timestamp,appid,player_count
0,2025-12-10 23:20:00,1566200,66
1,2025-12-10 23:20:00,1324350,2
2,2025-12-10 23:20:00,2494350,201
3,2025-12-10 23:20:00,853200,1
4,2025-12-10 23:20:00,301120,136


In [11]:
# Calculate average player count per game for popularity tiers
popularity_df = player_count_df.groupby('appid').agg({
    'player_count': ['mean', 'max', 'std']
}).reset_index()
popularity_df.columns = ['appid', 'avg_player_count', 'max_player_count', 'std_player_count']
popularity_df = popularity_df.fillna(0)

# Create popularity tiers based on average player count
popularity_df['popularity_tier'] = pd.qcut(
    popularity_df['avg_player_count'].rank(method='first'), 
    q=3, 
    labels=['low', 'medium', 'high']
)

print("Popularity tier distribution:")
print(popularity_df['popularity_tier'].value_counts())
print("\nPopularity stats:")
popularity_df.groupby('popularity_tier')['avg_player_count'].describe()

Popularity tier distribution:
popularity_tier
low       41
high      41
medium    40
Name: count, dtype: int64

Popularity stats:


,count,mean,std,min,25%,50%,75%,max
popularity_tier,,,,,,,,
low,41.0,1.981446,1.704338,0.000000,0.410714,1.860714,2.885714,6.167857
medium,40.0,22.121339,13.152873,6.557143,10.895536,18.560714,34.329464,48.635714
high,41.0,3702.175523,15236.208798,52.835714,99.242857,164.067857,450.842857,91406.960714


In [12]:
# Merge popularity data with games metadata
games_df = games_df.merge(popularity_df[['appid', 'avg_player_count', 'popularity_tier']], 
                          on='appid', how='left')

# Fill missing popularity data
games_df['avg_player_count'] = games_df['avg_player_count'].fillna(0)
games_df['popularity_tier'] = games_df['popularity_tier'].fillna('low')

# Parse release date and create era tiers
def parse_release_year(date_str):
    """Extract year from release date string."""
    if pd.isna(date_str):
        return None
    try:
        # Handle formats like "17 Nov, 2008" or "Nov 17, 2008"
        match = re.search(r'(\d{4})', str(date_str))
        if match:
            return int(match.group(1))
    except:
        pass
    return None

games_df['release_year'] = games_df['release_date'].apply(parse_release_year)

# Create release era tiers
def get_release_era(year):
    if pd.isna(year):
        return 'unknown'
    if year < 2020:
        return 'classic'
    elif year < 2024:
        return 'recent'
    else:
        return 'new'

games_df['release_era'] = games_df['release_year'].apply(get_release_era)

print("Release era distribution:")
print(games_df['release_era'].value_counts())
print("\nFree vs Paid:")
print(games_df['is_free'].value_counts())

Release era distribution:
release_era
new        41
classic    40
recent     37
unknown     2
Name: count, dtype: int64

Free vs Paid:
is_free
False    111
True       7
Name: count, dtype: int64


In [13]:
# Load all reviews and create a master dataframe
def load_all_reviews(reviews_path):
    """Load all review CSV files from all genre folders."""
    all_reviews = []
    genre_folders = [f for f in reviews_path.iterdir() if f.is_dir()]
    
    print(f"Found {len(genre_folders)} genre folders")
    
    for genre_folder in genre_folders:
        genre = genre_folder.name
        csv_files = list(genre_folder.glob('reviews_*.csv'))
        
        for csv_file in csv_files:
            try:
                df = pd.read_csv(csv_file)
                df['genre'] = genre
                df['source_file'] = csv_file.name
                all_reviews.append(df)
            except Exception as e:
                print(f"Error loading {csv_file}: {e}")
    
    return pd.concat(all_reviews, ignore_index=True)

print("Loading all reviews...")
reviews_df = load_all_reviews(REVIEWS_PATH)
print(f"\nTotal reviews loaded: {len(reviews_df):,}")
print(f"Columns: {reviews_df.columns.tolist()}")

Loading all reviews...
Found 12 genre folders

Total reviews loaded: 103,946
Columns: ['appid', 'recommendationid', 'review_text', 'voted_up', 'timestamp_created', 'votes_up', 'votes_funny', 'weighted_vote_score', 'comment_count', 'playtime_at_review_minutes', 'playtime_forever_minutes', 'steamid', 'genre', 'source_file']


In [14]:
# Basic review statistics
print("=" * 50)
print("REVIEW DATA SUMMARY")
print("=" * 50)
print(f"\nTotal reviews: {len(reviews_df):,}")
print(f"Unique games: {reviews_df['appid'].nunique()}")
print(f"Unique reviewers: {reviews_df['steamid'].nunique():,}")

print("\nSentiment distribution (voted_up):")
sentiment_dist = reviews_df['voted_up'].value_counts()
print(f"  Positive (True): {sentiment_dist.get(True, 0):,} ({sentiment_dist.get(True, 0)/len(reviews_df)*100:.1f}%)")
print(f"  Negative (False): {sentiment_dist.get(False, 0):,} ({sentiment_dist.get(False, 0)/len(reviews_df)*100:.1f}%)")

print("\nReviews per genre:")
print(reviews_df['genre'].value_counts())

REVIEW DATA SUMMARY

Total reviews: 103,946
Unique games: 120
Unique reviewers: 99,382

Sentiment distribution (voted_up):
  Positive (True): 70,633 (68.0%)
  Negative (False): 33,313 (32.0%)

Reviews per genre:
genre
exploration_open_world    15705
action_fps                12459
science_fiction           12263
rpg_action                11482
casual                    11104
survival                   9498
puzzle_matching            8853
anime                      7243
horror                     6419
sports                     3806
hidden_object              2571
visual_novel               2543
Name: count, dtype: int64


## 2. Quality Filtering

In [15]:
# Add review text length column
reviews_df['text_length'] = reviews_df['review_text'].astype(str).apply(len)

print("Review length statistics (before filtering):")
print(reviews_df['text_length'].describe())

print("\nReviews by length category:")
print(f"  Very short (< 50 chars): {(reviews_df['text_length'] < 50).sum():,}")
print(f"  Short (50-150 chars): {((reviews_df['text_length'] >= 50) & (reviews_df['text_length'] < 150)).sum():,}")
print(f"  Medium (150-400 chars): {((reviews_df['text_length'] >= 150) & (reviews_df['text_length'] < 400)).sum():,}")
print(f"  Long (400+ chars): {(reviews_df['text_length'] >= 400).sum():,}")

Review length statistics (before filtering):
count    103946.000000
mean        355.804870
std         737.525013
min           1.000000
25%          31.000000
50%         107.000000
75%         344.000000
max       15991.000000
Name: text_length, dtype: float64

Reviews by length category:
  Very short (< 50 chars): 34,669
  Short (50-150 chars): 24,936
  Medium (150-400 chars): 21,471
  Long (400+ chars): 22,870


In [16]:
def is_spam_or_template(text):
    """Detect spam, ASCII art, or template reviews."""
    if pd.isna(text):
        return True
    
    text = str(text)
    
    # ASCII art detection (lots of special characters)
    if text.count('/') > 10 or text.count('|') > 10 or text.count('\\') > 10:
        return True
    
    # Repetitive content detection
    words = text.split()
    if len(words) > 5:
        unique_ratio = len(set(words)) / len(words)
        if unique_ratio < 0.3:  # Very repetitive
            return True
    
    # Template/survey detection
    template_patterns = [
        r'☐|☑|✓|✗',  # Checkbox patterns
        r'\d+/10',   # Rating patterns like "7/10"
    ]
    for pattern in template_patterns:
        if re.search(pattern, text) and len(text) > 500:
            # Long template-style reviews
            if text.count('☐') + text.count('☑') > 5:
                return True
    
    return False

# Apply quality filters
print("Applying quality filters...")
original_count = len(reviews_df)

# Filter 1: Minimum length (50 characters)
reviews_df = reviews_df[reviews_df['text_length'] >= 50].copy()
print(f"After min length filter (50 chars): {len(reviews_df):,} ({len(reviews_df)/original_count*100:.1f}%)")

# Filter 2: Remove spam/templates
reviews_df['is_spam'] = reviews_df['review_text'].apply(is_spam_or_template)
reviews_df = reviews_df[~reviews_df['is_spam']].copy()
print(f"After spam filter: {len(reviews_df):,} ({len(reviews_df)/original_count*100:.1f}%)")

# Filter 3: Remove duplicates by recommendation ID
reviews_df = reviews_df.drop_duplicates(subset=['recommendationid']).copy()
print(f"After deduplication: {len(reviews_df):,} ({len(reviews_df)/original_count*100:.1f}%)")

print(f"\nTotal removed: {original_count - len(reviews_df):,} reviews")

Applying quality filters...
After min length filter (50 chars): 69,277 (66.6%)
After spam filter: 68,561 (66.0%)
After deduplication: 68,560 (66.0%)

Total removed: 35,386 reviews


In [17]:
# Sentiment distribution after filtering
print("Sentiment distribution after filtering:")
sentiment_dist = reviews_df['voted_up'].value_counts()
print(f"  Positive (True): {sentiment_dist.get(True, 0):,} ({sentiment_dist.get(True, 0)/len(reviews_df)*100:.1f}%)")
print(f"  Negative (False): {sentiment_dist.get(False, 0):,} ({sentiment_dist.get(False, 0)/len(reviews_df)*100:.1f}%)")

Sentiment distribution after filtering:
  Positive (True): 42,130 (61.4%)
  Negative (False): 26,430 (38.6%)


## 3. Create Stratification Buckets

In [18]:
# Create review length tiers (target: 20% short, 40% medium, 40% long)
def get_length_tier(length):
    if length < 150:
        return 'short'
    elif length < 400:
        return 'medium'
    else:
        return 'long'

reviews_df['length_tier'] = reviews_df['text_length'].apply(get_length_tier)

print("Length tier distribution:")
print(reviews_df['length_tier'].value_counts())
print("\nPercentages:")
print(reviews_df['length_tier'].value_counts(normalize=True) * 100)

Length tier distribution:
length_tier
short     24925
long      22197
medium    21438
Name: count, dtype: int64

Percentages:
length_tier
short     36.355018
long      32.376021
medium    31.268961
Name: proportion, dtype: float64


In [19]:
# Create playtime tiers (<2h, 2-10h, 10h+)
def get_playtime_tier(minutes):
    if pd.isna(minutes):
        return 'unknown'
    hours = minutes / 60
    if hours < 2:
        return 'short'
    elif hours < 10:
        return 'medium'
    else:
        return 'long'

reviews_df['playtime_tier'] = reviews_df['playtime_at_review_minutes'].apply(get_playtime_tier)

print("Playtime tier distribution:")
print(reviews_df['playtime_tier'].value_counts())
print("\nPercentages:")
print(reviews_df['playtime_tier'].value_counts(normalize=True) * 100)

Playtime tier distribution:
playtime_tier
long       31091
medium     24847
short      12564
unknown       58
Name: count, dtype: int64

Percentages:
playtime_tier
long       45.348600
medium     36.241249
short      18.325554
unknown     0.084597
Name: proportion, dtype: float64


In [20]:
# Merge game metadata with reviews
reviews_df = reviews_df.merge(
    games_df[['appid', 'name', 'primary_genre_query', 'is_free', 'popularity_tier', 'release_era']],
    on='appid',
    how='left'
)

# Use primary_genre_query as the main genre (from metadata)
# Fall back to the folder-based genre if metadata is missing
reviews_df['primary_genre'] = reviews_df['primary_genre_query'].fillna(reviews_df['genre'])

print("Reviews with metadata merged:")
print(f"Total reviews: {len(reviews_df):,}")
print("\nPrimary genre distribution:")
print(reviews_df['primary_genre'].value_counts())

Reviews with metadata merged:
Total reviews: 68,560

Primary genre distribution:
primary_genre
science_fiction           9298
exploration_open_world    9236
action_fps                7968
rpg_action                7771
casual                    7211
survival                  6764
puzzle_matching           5828
anime                     4819
horror                    3956
sports                    2559
hidden_object             1577
visual_novel              1573
Name: count, dtype: int64


## 4. Analyze Available Reviews Per Game

In [21]:
# Analyze positive/negative review availability per game
game_review_stats = reviews_df.groupby('appid').agg({
    'voted_up': ['sum', 'count'],
    'name': 'first',
    'primary_genre': 'first',
    'popularity_tier': 'first',
    'is_free': 'first',
    'release_era': 'first'
}).reset_index()

game_review_stats.columns = ['appid', 'positive_count', 'total_count', 'name', 
                              'primary_genre', 'popularity_tier', 'is_free', 'release_era']
game_review_stats['negative_count'] = game_review_stats['total_count'] - game_review_stats['positive_count']
game_review_stats['positive_ratio'] = game_review_stats['positive_count'] / game_review_stats['total_count']

# Calculate maximum balanced samples possible per game
game_review_stats['max_balanced_per_class'] = game_review_stats[['positive_count', 'negative_count']].min(axis=1)

print("Game review statistics:")
print(game_review_stats[['name', 'total_count', 'positive_count', 'negative_count', 'positive_ratio', 'max_balanced_per_class']].head(20))

Game review statistics:
                                                 name  total_count  \
0                                         Left 4 Dead          947   
1                                           BioShock™         1265   
2                                    Star Trek Online         1428   
3                                        World of Goo          750   
4   The Elder Scrolls IV: Oblivion® Game of the Ye...         1221   
5                                          Samorost 2          742   
6      Batman: Arkham City - Game of the Year Edition         1164   
7                                         To the Moon         1437   
8                                              Lucius         1219   
9   THE KING OF FIGHTERS '98 ULTIMATE MATCH FINAL ...          551   
10                           Far Cry 3 - Blood Dragon         1375   
11                                     Battle Nations          709   
12        RollerCoaster Tycoon® 2: Triple Thrill Pack          780

In [22]:
# Identify games with very few negative reviews
print("Games with limited negative reviews (< 100):")
limited_negatives = game_review_stats[game_review_stats['negative_count'] < 100].sort_values('negative_count')
print(limited_negatives[['name', 'primary_genre', 'positive_count', 'negative_count', 'positive_ratio']])

print(f"\n{len(limited_negatives)} games have < 100 negative reviews")
print(f"\nTotal available balanced reviews: {game_review_stats['max_balanced_per_class'].sum() * 2:,}")

Games with limited negative reviews (< 100):
                                       name  primary_genre  positive_count  \
53                TrymenT ―献给渴望改变的你― AlphA篇   visual_novel               8   
95                      Tenioha! feat. Mami   visual_novel               5   
94                          Kuroinu 2 Redux   visual_novel               8   
98   Climb Challenge - Find Items Cyberpunk         sports               8   
111               Lateral Thinking Together  hidden_object               1   
..                                      ...            ...             ...   
65                 The Jackbox Party Pack 8         casual             492   
44                 The Jackbox Party Pack 6         casual             386   
22                                 Overload     action_fps             813   
58                        Turbo Golf Racing         sports             425   
39                         Prison Simulator     action_fps             322   

     negative_coun

## 5. Select 50 Games via Stratified Sampling

In [23]:
# Filter games with at least 50 reviews per class (minimum viable for stratified sampling)
MIN_REVIEWS_PER_CLASS = 50
eligible_games = game_review_stats[game_review_stats['max_balanced_per_class'] >= MIN_REVIEWS_PER_CLASS].copy()

print(f"Games eligible for selection (>= {MIN_REVIEWS_PER_CLASS} reviews per class): {len(eligible_games)}")
print("\nEligible games by genre:")
print(eligible_games['primary_genre'].value_counts())

Games eligible for selection (>= 50 reviews per class): 64

Eligible games by genre:
primary_genre
exploration_open_world    10
action_fps                 8
rpg_action                 7
casual                     7
science_fiction            7
puzzle_matching            6
survival                   6
sports                     4
anime                      4
horror                     2
hidden_object              2
visual_novel               1
Name: count, dtype: int64


In [24]:
def stratified_game_selection(games_df, n_games=50, seed=42):
    """
    Select n_games using stratified sampling across:
    - Genre (primary factor)
    - Popularity tier
    - Free/Paid status
    - Release era
    """
    np.random.seed(seed)
    
    selected_games = []
    genres = games_df['primary_genre'].unique()
    n_genres = len(genres)
    
    # Target ~4 games per genre (50 / 12 ≈ 4)
    base_per_genre = n_games // n_genres
    remainder = n_games % n_genres
    
    print(f"Selecting {n_games} games from {n_genres} genres")
    print(f"Base per genre: {base_per_genre}, remainder: {remainder}")
    
    for i, genre in enumerate(genres):
        genre_games = games_df[games_df['primary_genre'] == genre].copy()
        
        # Add 1 extra game to first 'remainder' genres
        n_select = base_per_genre + (1 if i < remainder else 0)
        n_select = min(n_select, len(genre_games))  # Can't select more than available
        
        if len(genre_games) <= n_select:
            # Take all games if not enough
            selected = genre_games
        else:
            # Try to balance by popularity and is_free
            # Score games to prefer diversity
            genre_games['selection_score'] = (
                genre_games['max_balanced_per_class'] * 0.5 +  # Prefer games with more balanced reviews
                np.random.rand(len(genre_games)) * 100  # Random component
            )
            
            # Try to get mix of popularity tiers
            selected_list = []
            for tier in ['low', 'medium', 'high']:
                tier_games = genre_games[genre_games['popularity_tier'] == tier]
                if len(tier_games) > 0:
                    n_from_tier = max(1, n_select // 3)
                    tier_sample = tier_games.nlargest(n_from_tier, 'selection_score')
                    selected_list.append(tier_sample)
            
            if selected_list:
                selected = pd.concat(selected_list).drop_duplicates()
                # If we need more, add from remaining
                if len(selected) < n_select:
                    remaining = genre_games[~genre_games['appid'].isin(selected['appid'])]
                    additional = remaining.nlargest(n_select - len(selected), 'selection_score')
                    selected = pd.concat([selected, additional])
                selected = selected.head(n_select)
            else:
                selected = genre_games.nlargest(n_select, 'selection_score')
        
        selected_games.append(selected)
        print(f"  {genre}: selected {len(selected)} games")
    
    result = pd.concat(selected_games, ignore_index=True)
    return result

# Select 50 games
selected_games = stratified_game_selection(eligible_games, n_games=50, seed=RANDOM_SEED)
print(f"\nTotal selected games: {len(selected_games)}")

Selecting 50 games from 12 genres
Base per genre: 4, remainder: 2
  action_fps: selected 5 games
  exploration_open_world: selected 5 games
  puzzle_matching: selected 4 games
  rpg_action: selected 4 games
  casual: selected 4 games
  horror: selected 2 games
  science_fiction: selected 4 games
  sports: selected 4 games
  survival: selected 4 games
  hidden_object: selected 2 games
  anime: selected 4 games
  visual_novel: selected 1 games

Total selected games: 43


In [ ]:
# Display selected games summary
print("=" * 60)
print("SELECTED GAMES SUMMARY")
print("=" * 60)

print("\nBy Genre:")
print(selected_games['primary_genre'].value_counts())

print("\nBy Popularity Tier:")
print(selected_games['popularity_tier'].value_counts())

print("\nBy Release Era:")
print(selected_games['release_era'].value_counts())

print("\nBy Free/Paid:")
print(selected_games['is_free'].value_counts())

print(f"\nTotal balanced reviews available: {selected_games['max_balanced_per_class'].sum() * 2:,}")

SELECTED GAMES SUMMARY

By Genre:
primary_genre
action_fps                5
exploration_open_world    5
puzzle_matching           4
rpg_action                4
casual                    4
science_fiction           4
sports                    4
survival                  4
anime                     4
horror                    2
hidden_object             2
visual_novel              1
Name: count, dtype: int64

By Popularity Tier:
popularity_tier
high      22
medium    13
low        8
Name: count, dtype: int64

By Release Era:
release_era
classic    22
recent     13
new         7
unknown     1
Name: count, dtype: int64

By Free/Paid:
is_free
False    40
True      2
Name: count, dtype: int64

Total balanced reviews available: 29,666


In [26]:
# Display full list of selected games
print("\nSelected Games List:")
display_cols = ['appid', 'name', 'primary_genre', 'popularity_tier', 'is_free', 
                'positive_count', 'negative_count', 'max_balanced_per_class']
selected_games[display_cols].sort_values('primary_genre')


Selected Games List:


,appid,name,primary_genre,popularity_tier,is_free,positive_count,negative_count,max_balanced_per_class
0,541200,GTTOD: Get To The Orange Door,action_fps,low,False,537,281,281
1,7670,BioShock™,action_fps,medium,False,487,778,487
2,973580,Sniper Ghost Warrior Contracts,action_fps,high,False,520,828,520
3,500,Left 4 Dead,action_fps,high,False,358,589,358
4,751630,After the Fall®,action_fps,medium,False,688,363,363
40,1353230,Bomb Rush Cyberfunk,anime,high,False,535,215,215
39,1105510,Yakuza 5 Remastered,anime,high,False,517,245,245
38,589530,Hakuoki: Kyoto Winds,anime,medium,False,454,53,53
41,1858630,SWORD ART ONLINE Fractured Daydream,anime,high,False,440,259,259
20,2296990,We Were Here Expeditions: The FriendShip,casual,medium,False,589,338,338


## 6. Sample Reviews with Flexible Per-Game Quotas

In [27]:
# Configuration
TARGET_TOTAL_REVIEWS = 50000
TARGET_PER_CLASS = TARGET_TOTAL_REVIEWS // 2  # 25,000 positive, 25,000 negative

# Target distribution for length tiers (within each sentiment class)
LENGTH_TIER_TARGETS = {'short': 0.20, 'medium': 0.40, 'long': 0.40}

# Target distribution for playtime tiers
PLAYTIME_TIER_TARGETS = {'short': 0.33, 'medium': 0.33, 'long': 0.34}

print(f"Target: {TARGET_TOTAL_REVIEWS:,} total reviews")
print(f"  - {TARGET_PER_CLASS:,} positive")
print(f"  - {TARGET_PER_CLASS:,} negative")

Target: 50,000 total reviews
  - 25,000 positive
  - 25,000 negative


In [ ]:
def stratified_review_sampling(reviews_df, selected_game_ids, target_per_class=25000, seed=42):
    """
    Sample reviews with flexible per-game quotas using stratified sampling.
    Balances across sentiment, length tier, and playtime tier.
    """
    np.random.seed(seed)
    
    # Filter to selected games only
    eligible_reviews = reviews_df[reviews_df['appid'].isin(selected_game_ids)].copy()
    print(f"Eligible reviews from {len(selected_game_ids)} games: {len(eligible_reviews):,}")
    
    # Separate by sentiment
    positive_reviews = eligible_reviews[eligible_reviews['voted_up']].copy()
    negative_reviews = eligible_reviews[~eligible_reviews['voted_up']].copy()
    
    print(f"Available positive: {len(positive_reviews):,}")
    print(f"Available negative: {len(negative_reviews):,}")
    
    def sample_with_stratification(df, n_target, length_targets, playtime_targets):
        """Sample from a dataframe with stratification by length and playtime tiers."""
        sampled = []
        
        # Calculate samples needed per length tier
        for length_tier, length_ratio in length_targets.items():
            length_subset = df[df['length_tier'] == length_tier]
            n_from_length = int(n_target * length_ratio)
            
            if len(length_subset) == 0:
                continue
            
            # Within each length tier, stratify by playtime
            for playtime_tier, playtime_ratio in playtime_targets.items():
                playtime_subset = length_subset[length_subset['playtime_tier'] == playtime_tier]
                n_from_playtime = int(n_from_length * playtime_ratio)
                
                if len(playtime_subset) == 0:
                    continue
                
                # Sample
                n_sample = min(n_from_playtime, len(playtime_subset))
                if n_sample > 0:
                    sample = playtime_subset.sample(n=n_sample, random_state=seed)
                    sampled.append(sample)
        
        if not sampled:
            return pd.DataFrame()
        
        result = pd.concat(sampled, ignore_index=True)
        
        # If we didn't get enough, sample more randomly from remaining
        if len(result) < n_target:
            remaining = df[~df['recommendationid'].isin(result['recommendationid'])]
            n_additional = min(n_target - len(result), len(remaining))
            if n_additional > 0:
                additional = remaining.sample(n=n_additional, random_state=seed)
                result = pd.concat([result, additional], ignore_index=True)
        
        return result.head(n_target)
    
    # Sample positive and negative reviews
    print("\nSampling positive reviews...")
    sampled_positive = sample_with_stratification(
        positive_reviews, target_per_class, LENGTH_TIER_TARGETS, PLAYTIME_TIER_TARGETS
    )
    print(f"  Sampled: {len(sampled_positive):,}")
    
    print("\nSampling negative reviews...")
    sampled_negative = sample_with_stratification(
        negative_reviews, target_per_class, LENGTH_TIER_TARGETS, PLAYTIME_TIER_TARGETS
    )
    print(f"  Sampled: {len(sampled_negative):,}")
    
    # Combine
    final_sample = pd.concat([sampled_positive, sampled_negative], ignore_index=True)
    
    return final_sample

# Sample reviews
sampled_reviews = stratified_review_sampling(
    reviews_df, 
    selected_games['appid'].tolist(),
    target_per_class=TARGET_PER_CLASS,
    seed=RANDOM_SEED
)

print("\n" + "=" * 50)
print(f"FINAL SAMPLE: {len(sampled_reviews):,} reviews")
print("=" * 50)

Eligible reviews from 43 games: 41,669
Available positive: 23,003
Available negative: 18,666

Sampling positive reviews...
  Sampled: 23,003

Sampling negative reviews...
  Sampled: 18,666

FINAL SAMPLE: 41,669 reviews


In [ ]:
# Verify sampling distribution
print("SAMPLING DISTRIBUTION VERIFICATION")
print("=" * 50)

print("\n1. Sentiment Balance:")
sentiment_counts = sampled_reviews['voted_up'].value_counts()
print(f"   Positive: {sentiment_counts.get(True, 0):,} ({sentiment_counts.get(True, 0)/len(sampled_reviews)*100:.1f}%)")
print(f"   Negative: {sentiment_counts.get(False, 0):,} ({sentiment_counts.get(False, 0)/len(sampled_reviews)*100:.1f}%)")

print("\n2. Length Tier Distribution:")
length_counts = sampled_reviews['length_tier'].value_counts(normalize=True) * 100
for tier in ['short', 'medium', 'long']:
    actual = length_counts.get(tier, 0)
    target = LENGTH_TIER_TARGETS.get(tier, 0) * 100
    print(f"   {tier}: {actual:.1f}% (target: {target:.0f}%)")

print("\n3. Playtime Tier Distribution:")
playtime_counts = sampled_reviews['playtime_tier'].value_counts(normalize=True) * 100
for tier in ['short', 'medium', 'long']:
    actual = playtime_counts.get(tier, 0)
    target = PLAYTIME_TIER_TARGETS.get(tier, 0) * 100
    print(f"   {tier}: {actual:.1f}% (target: {target:.0f}%)")

print("\n4. Genre Distribution:")
print(sampled_reviews['primary_genre'].value_counts())

print(f"\n5. Games Represented: {sampled_reviews['appid'].nunique()}")

SAMPLING DISTRIBUTION VERIFICATION

1. Sentiment Balance:
   Positive: 23,003 (55.2%)
   Negative: 18,666 (44.8%)

2. Length Tier Distribution:
   short: 35.9% (target: 20%)
   medium: 31.0% (target: 40%)
   long: 33.1% (target: 40%)

3. Playtime Tier Distribution:
   short: 17.6% (target: 33%)
   medium: 35.7% (target: 33%)
   long: 46.6% (target: 34%)

4. Genre Distribution:
primary_genre
science_fiction           6033
action_fps                5429
exploration_open_world    5158
survival                  4436
casual                    4124
rpg_action                4021
puzzle_matching           3725
anime                     2718
sports                    2326
horror                    2193
hidden_object              894
visual_novel               612
Name: count, dtype: int64

5. Games Represented: 43


## 7. Text Preprocessing

In [ ]:
def preprocess_text(text):
    """
    Preprocess review text for sentiment analysis.
    - Lowercase
    - Remove URLs
    - Remove HTML tags
    - Remove excessive punctuation (keep some for sentiment)
    - Normalize whitespace
    """
    if pd.isna(text):
        return ""
    
    text = str(text)
    
    # Lowercase
    text = text.lower()
    
    # Remove URLs
    text = re.sub(r'http\S+|www\.\S+', '', text)
    
    # Remove HTML tags
    text = re.sub(r'<[^>]+>', '', text)
    
    # Remove excessive punctuation (more than 3 consecutive)
    text = re.sub(r'([!?.]){3,}', r'\1\1\1', text)
    
    # Remove special characters but keep basic punctuation
    text = re.sub(r'[^\w\s.,!?\'-]', ' ', text)
    
    # Normalize whitespace
    text = re.sub(r'\s+', ' ', text).strip()
    
    return text

# Apply preprocessing
print("Preprocessing text...")
sampled_reviews['processed_text'] = sampled_reviews['review_text'].apply(preprocess_text)

# Show examples
print("\nPreprocessing examples:")
for i in range(3):
    print(f"\n--- Example {i+1} ---")
    print(f"Original: {sampled_reviews.iloc[i]['review_text'][:200]}...")
    print(f"Processed: {sampled_reviews.iloc[i]['processed_text'][:200]}...")

Preprocessing text...

Preprocessing examples:

--- Example 1 ---
Original: jeigu norit susitraumuot, patariu sita zaidima pazaist bent 15min...
Processed: jeigu norit susitraumuot, patariu sita zaidima pazaist bent 15min...

--- Example 2 ---
Original: What's here, I really like. Still too buggy in it's current state though. Will update....
Processed: what's here, i really like. still too buggy in it's current state though. will update....

--- Example 3 ---
Original: This game gets exponentially harder when your friend is missing brain cells and has an extra chromosome....
Processed: this game gets exponentially harder when your friend is missing brain cells and has an extra chromosome....


In [31]:
# Remove reviews that became too short after preprocessing
sampled_reviews['processed_length'] = sampled_reviews['processed_text'].apply(len)

before_count = len(sampled_reviews)
sampled_reviews = sampled_reviews[sampled_reviews['processed_length'] >= 30].copy()
after_count = len(sampled_reviews)

print(f"Removed {before_count - after_count} reviews with processed text < 30 chars")
print(f"Final dataset size: {len(sampled_reviews):,}")

Removed 90 reviews with processed text < 30 chars
Final dataset size: 41,579


## 8. Create Train/Validation/Test Splits

In [33]:
from sklearn.model_selection import train_test_split

# Create stratified split: 70% train, 15% validation, 15% test
# Stratify by sentiment and genre

# Create stratification key
sampled_reviews['stratify_key'] = (
    sampled_reviews['voted_up'].astype(str) + '_' + 
    sampled_reviews['primary_genre'].astype(str)
)

# First split: 70% train, 30% temp
train_df, temp_df = train_test_split(
    sampled_reviews,
    test_size=0.30,
    stratify=sampled_reviews['stratify_key'],
    random_state=RANDOM_SEED
)

# Second split: 50% of temp = 15% validation, 50% of temp = 15% test
val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    stratify=temp_df['stratify_key'],
    random_state=RANDOM_SEED
)

print("Dataset Splits:")
print(f"  Train: {len(train_df):,} ({len(train_df)/len(sampled_reviews)*100:.1f}%)")
print(f"  Validation: {len(val_df):,} ({len(val_df)/len(sampled_reviews)*100:.1f}%)")
print(f"  Test: {len(test_df):,} ({len(test_df)/len(sampled_reviews)*100:.1f}%)")

Dataset Splits:
  Train: 29,105 (70.0%)
  Validation: 6,237 (15.0%)
  Test: 6,237 (15.0%)


In [34]:
# Verify splits maintain balance
print("Sentiment balance verification:")
for name, df in [('Train', train_df), ('Validation', val_df), ('Test', test_df)]:
    pos_pct = df['voted_up'].mean() * 100
    print(f"  {name}: {pos_pct:.1f}% positive, {100-pos_pct:.1f}% negative")

Sentiment balance verification:
  Train: 55.1% positive, 44.9% negative
  Validation: 55.1% positive, 44.9% negative
  Test: 55.2% positive, 44.8% negative


## 9. Save Processed Datasets

In [35]:
# Create output directory
OUTPUT_DIR = Path('../processed_data')
OUTPUT_DIR.mkdir(exist_ok=True)

# Select columns to save
SAVE_COLUMNS = [
    'recommendationid',
    'appid',
    'review_text',
    'processed_text',
    'voted_up',
    'playtime_at_review_minutes',
    'playtime_tier',
    'text_length',
    'length_tier',
    'primary_genre',
    'name'
]

# Save datasets
train_df[SAVE_COLUMNS].to_csv(OUTPUT_DIR / 'train.csv', index=False)
val_df[SAVE_COLUMNS].to_csv(OUTPUT_DIR / 'validation.csv', index=False)
test_df[SAVE_COLUMNS].to_csv(OUTPUT_DIR / 'test.csv', index=False)

# Save full dataset as well
sampled_reviews[SAVE_COLUMNS].to_csv(OUTPUT_DIR / 'full_dataset.csv', index=False)

print(f"Datasets saved to {OUTPUT_DIR.absolute()}")
print(f"  - train.csv: {len(train_df):,} reviews")
print(f"  - validation.csv: {len(val_df):,} reviews")
print(f"  - test.csv: {len(test_df):,} reviews")
print(f"  - full_dataset.csv: {len(sampled_reviews):,} reviews")

Datasets saved to d:\Khai\Code\Projects\WebMining\notebooks\..\processed_data
  - train.csv: 29,105 reviews
  - validation.csv: 6,237 reviews
  - test.csv: 6,237 reviews
  - full_dataset.csv: 41,579 reviews


In [36]:
# Save selected games metadata
selected_games.to_csv(OUTPUT_DIR / 'selected_games.csv', index=False)
print(f"  - selected_games.csv: {len(selected_games)} games")

# Save preprocessing summary
summary = {
    'total_reviews': len(sampled_reviews),
    'train_size': len(train_df),
    'val_size': len(val_df),
    'test_size': len(test_df),
    'num_games': sampled_reviews['appid'].nunique(),
    'num_genres': sampled_reviews['primary_genre'].nunique(),
    'positive_ratio': sampled_reviews['voted_up'].mean(),
    'random_seed': RANDOM_SEED,
    'min_text_length': 50,
    'length_tier_targets': LENGTH_TIER_TARGETS,
    'playtime_tier_targets': PLAYTIME_TIER_TARGETS
}

import json
with open(OUTPUT_DIR / 'preprocessing_summary.json', 'w') as f:
    json.dump(summary, f, indent=2)

print("  - preprocessing_summary.json")
print("\nPreprocessing complete!")

  - selected_games.csv: 43 games
  - preprocessing_summary.json

Preprocessing complete!


## Summary

### What was done:
1. **Loaded** ~300,000+ reviews from 120 games across 12 genres
2. **Quality filtered** to remove short (<50 chars), spam, and duplicate reviews
3. **Created stratification buckets** for:
   - Review length: short (50-150), medium (150-400), long (400+)
   - Playtime: short (<2h), medium (2-10h), long (10h+)
   - Genre, popularity, release era
4. **Selected 50 games** via stratified sampling across genres and other features
5. **Sampled ~50,000 reviews** with balanced sentiment (50/50) and stratified length/playtime
6. **Preprocessed text**: lowercase, removed URLs/HTML, normalized whitespace
7. **Split** into train (70%), validation (15%), test (15%) with stratification

### Output files:
- `processed_data/train.csv`
- `processed_data/validation.csv`  
- `processed_data/test.csv`
- `processed_data/full_dataset.csv`
- `processed_data/selected_games.csv`
- `processed_data/preprocessing_summary.json`